# Main results

## 1. Build model

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# Filter out distutils warning in pd.datareader
import warnings
warnings.filterwarnings(
    "ignore",
    message="distutils Version classes are deprecated.*",
    category=DeprecationWarning,
)


In [10]:
from src import (
    download_save_raw_data,
    clean_save_data,
    create_save_portfolios,
    build_save_model,
)


from configs import PROJ_CONFIG

### 1.a. Download data (Resource intensive)

In [11]:
download_save_raw_data(PROJ_CONFIG)

2026-04-11 14:08:12 | +  96.442s | INFO | Starting to download all necessary data for the project...
--------------------------------------------------------------------------------


Loading library list...


2026-04-11 14:08:16 | +   4.138s | INFO | Successfully connected to WRDS database


Done


2026-04-11 14:08:23 | +   6.980s | DEBUG | Downloaded monthly prices for the observable universe of stocks from WRDS from 2009-05-01 to 2009-12-31
2026-04-11 14:08:32 | +   9.525s | DEBUG | Downloaded monthly prices for the observable universe of stocks from WRDS from 2010-01-01 to 2010-12-31
2026-04-11 14:08:43 | +  11.099s | DEBUG | Downloaded monthly prices for the observable universe of stocks from WRDS from 2011-01-01 to 2011-12-31
2026-04-11 14:08:55 | +  11.819s | DEBUG | Downloaded monthly prices for the observable universe of stocks from WRDS from 2012-01-01 to 2012-12-31
2026-04-11 14:09:08 | +  13.325s | DEBUG | Downloaded monthly prices for the observable universe of stocks from WRDS from 2013-01-01 to 2013-12-31
2026-04-11 14:09:23 | +  14.497s | DEBUG | Downloaded monthly prices for the observable universe of stocks from WRDS from 2014-01-01 to 2014-12-31
2026-04-11 14:09:37 | +  14.149s | DEBUG | Downloaded monthly prices for the observable universe of stocks from WRDS f

### 1.b. Process data (Resource intensive)

In [12]:
clean_save_data(PROJ_CONFIG)

2026-04-11 14:13:27 | +   0.069s | INFO | Starting importing raw data....
--------------------------------------------------------------------------------
2026-04-11 14:13:28 | +   0.901s | INFO | Successfully downloaded all raw data
2026-04-11 14:13:28 | +   0.001s | INFO | Starting data cleaning process....
--------------------------------------------------------------------------------
2026-04-11 14:13:28 | +   0.001s | INFO | Cleaned the factor data
2026-04-11 14:13:28 | +   0.002s | DEBUG | Monthly factor data sample:

            Mkt-RF     SMB     HML     RMW     CMA      RF
date                                                      
2009-06-01  0.0042  0.0227 -0.0276 -0.0141 -0.0037  0.0001
2009-07-01  0.0774  0.0231  0.0489 -0.0026  0.0308  0.0001
2009-08-01  0.0333 -0.0010  0.0760 -0.0285  0.0330  0.0001
2009-09-01  0.0408  0.0272  0.0115  0.0115  0.0034  0.0001
2009-10-01 -0.0255 -0.0485 -0.0417  0.0422 -0.0148  0.0000

2026-04-11 14:13:28 | +   0.031s | INFO | Finished clean

### 1.c. Create the portfolios

In [13]:
create_save_portfolios(PROJ_CONFIG)

2026-04-11 14:13:32 | +   0.135s | INFO | Starting to download processed data...
--------------------------------------------------------------------------------
2026-04-11 14:13:33 | +   0.679s | INFO | Successfully downloaded the processed data
2026-04-11 14:13:33 | +   0.001s | INFO | Starting to create portfolios....
--------------------------------------------------------------------------------
2026-04-11 14:13:33 | +   0.297s | INFO | Successfully assigned industries to firms according to the Fama-French industry portfolios
2026-04-11 14:14:04 | +  31.267s | INFO | Dropped non-significant portfolios with less than 10 firms.            
Number of portfolios dropped: 35 (144->109)
2026-04-11 14:14:04 | +   0.011s | INFO | Dropped portfolios with less than 26 occurances over entire period.
2026-04-11 14:14:05 | +   0.027s | INFO | Dropped portfolios with less than 26 occurances for each subperiod period.
2026-04-11 14:14:05 | +   0.002s | INFO | Dropped all non-significant portfoli

### 1.d. Create the model and run the regression

In [15]:
build_save_model(PROJ_CONFIG)

2026-04-11 14:14:11 | +   3.347s | INFO | Starting to download portfolio data...
--------------------------------------------------------------------------------
2026-04-11 14:14:11 | +   0.289s | INFO | Downloaded portfolio data successfully
2026-04-11 14:14:11 | +   0.002s | INFO | Starting analysis of the entire period...
--------------------------------------------------------------------------------
2026-04-11 14:14:11 | +   0.178s | INFO | Extracted factor loadings successfully for multiple stocks
2026-04-11 14:14:11 | +   0.050s | INFO | Predicted returns successfully using the factor model
2026-04-11 14:14:11 | +   0.003s | INFO | Compared predicted and actual returns successfully
2026-04-11 14:14:11 | +   0.001s | INFO | Tested the significance of model parameters successfully
2026-04-11 14:14:11 | +   0.001s | INFO | Successfully analysed portfolios for the entire period
2026-04-11 14:14:11 | +   0.001s | INFO | Starting Analysis for subperiods...
----------------------------

# 2. Data Analysis

In [39]:
from pathlib import Path
import sys
import pandas as pd
from typing import List, Dict, Tuple
# Set matlplotlib to inline mode
%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [51]:
# Import the data
from src.utils import import_analysis_data

from configs import (
    ANALYSIS_PATHS,
)

# Import files and set index
monthly_factor_loadings, monthly_predicted_returns, factor_loadings_overtime = (
    import_analysis_data()
)

## Analysis of the changes in Beta

In [46]:
# Filter out only the betas
betas_over_time: pd.DataFrame = factor_loadings_overtime.loc["Beta"].swaplevel(axis=1)

# Get the different timeframes
timeframes: pd.Index = betas_over_time.columns.levels[0]
portfolios: pd.Index = betas_over_time.columns.levels[1]

In [47]:
# Get the changes in betas compared to the entire period
changes_betas_entire_period = pd.DataFrame()
for timeframe in timeframes:
    if timeframe != "Entire Period":
        diff = betas_over_time.loc[:, timeframe] - betas_over_time.loc[:, "Entire Period"]
        diff.columns = pd.MultiIndex.from_product([[timeframe], diff.columns])
        changes_betas_entire_period = pd.concat([changes_betas_entire_period, diff], axis=1)


# Get the changes in betas compared to the previous period
ordered_timeframe: List[str] = ['07/2009:01/2015', '01/2015:02/2020', '02/2020:12/2023','12/2023:01/2026']
changes_betas_period_wise = pd.DataFrame()
for i in range(1, len(ordered_timeframe)):
    current_timeframe = ordered_timeframe[i]
    previous_timeframe = ordered_timeframe[i-1]
    diff = betas_over_time.loc[:, current_timeframe] - betas_over_time.loc[:, previous_timeframe]
    diff.columns = pd.MultiIndex.from_product([[f"{previous_timeframe}->{current_timeframe}"], diff.columns])
    changes_betas_period_wise = pd.concat([changes_betas_period_wise, diff], axis=1)

In [48]:
# Desciptors
descriptors: Tuple[str] = ("Mean Absolute Change", "Mean Change", "Max Change", "Min Change")

def get_description_stats(df: pd.DataFrame) -> pd.DataFrame:

    # Calculate the mean absolute change for each portfolio and timeframe
    df.loc["Mean beta changes",:] = df.loc[["CMA", "HML", "Mkt-RF", "RMW", "SMB"]].abs().mean()

    # Calculate the mean squared change in each timeperiod for comparability
    for timeframe in df.columns.levels[0]:
        df[timeframe, "Mean Absolute Change"] = (
            df[timeframe].abs().mean(axis=1)
        )
        df[timeframe, "Mean Change"] = (
            df[timeframe].mean(axis=1)
        )
        df[timeframe, "Max Change"] = (
            df[timeframe].max(axis=1)
        )
        df[timeframe, "Min Change"] = (
            df[timeframe].min(axis=1)
        )
    
    return df

changes_betas_entire_period = get_description_stats(changes_betas_entire_period)
changes_betas_period_wise   = get_description_stats(changes_betas_period_wise)

In [49]:
def split_data_for_analysis(df: pd.DataFrame)-> Dict[str, pd.DataFrame]:
    # Create the dictionary to hold the data
    data_dict = {"entire data": df}

    # Descriptions
    data_dict["descriptions"] = df.loc[:, df.columns.get_level_values(1).isin(descriptors)]

    # Main Portfolios
    data_dict["main_portfolios"] = df.loc[:, ~df.columns.get_level_values(1).str.contains("-") & ~df.columns.get_level_values(1).isin(descriptors)]

    # Sub portfolios
    data_dict["subportfolios"] = df.loc[:, df.columns.get_level_values(1).str.contains("-")]

    return data_dict

data_period_wise: Dict[str, pd.DataFrame] = split_data_for_analysis(changes_betas_period_wise)
data_compared_to_entire_period: Dict[str, pd.DataFrame] = split_data_for_analysis(changes_betas_entire_period)

In [52]:
# Save the data
for name, df in data_period_wise.items():
    df.to_csv(ANALYSIS_PATHS.RESULT_TABLES_DIR / f"changes_betas_period_wise_{name}.csv")

for name, df in data_compared_to_entire_period.items():
    df.to_csv(ANALYSIS_PATHS.RESULT_TABLES_DIR / f"changes_betas_compared_to_entire_period_{name}.csv")
